# Upload vers Supabase Storage + import `transaction` (CSV RLM / Farfetch / DTC)

1. **Storage** : envoi du fichier dans un bucket (CSV, ou autre).
2. **Base** (optionnel) : pour un **CSV** dont les en-têtes correspondent au modèle Power Query (colonnes sources puis harmonisation), les lignes sont transformées comme dans votre requête M puis insérées dans la table PostgreSQL `transaction` (même schéma que `api/lib/process_transactions.py`).

Les jointures Power Query **Customers Categorization RLM** et **Products_with_unique_upc** sont optionnelles : `location_id` / `product_id` peuvent rester vides si non fournis.

In [1]:
%pip install "typing-extensions>=4.15.0" supabase python-dotenv pandas -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from supabase import create_client

load_dotenv()

raw_url = os.environ["SUPABASE_URL"]
match = re.search(r"([a-z]{20,})\.supabase\.co", raw_url)
project_ref = match.group(1) if match else None
assert project_ref, f"Impossible d'extraire le project ref depuis SUPABASE_URL ({raw_url})"

api_url = f"https://{project_ref}.supabase.co"

assert "SUPABASE_SERVICE_ROLE_KEY" in os.environ, (
    "SUPABASE_SERVICE_ROLE_KEY manquante dans .env\n"
    "→ Dashboard Supabase > Settings > API > service_role key"
)
api_key = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

supabase = create_client(api_url, api_key)
print(f"Supabase client ready : {api_url}")

Supabase client ready : https://nybxcfjjnkgxzgaeitlk.supabase.co


In [3]:
buckets = supabase.storage.list_buckets()
print("Buckets disponibles :")
for b in buckets:
    print(f"  - {b.name}  (public={b.public})")

Buckets disponibles :
  - farfetch_files  (public=False)
  - TEST  (public=False)


In [ ]:
# ---- Configuration ----
LOCAL_FILE = ""          # ex: "/Users/me/data/export.csv"
BUCKET     = "farfetch_files"   # ex: "farfetch_files"
DEST_PATH  = ""          # ex: "imports/2025/export.csv" (vide = nom du fichier local)

# Storage
UPLOAD_TO_STORAGE = True
STORAGE_UPSERT = True   # remplace le fichier si le chemin existe déjà

# Base : uniquement si le fichier est un .csv
INSERT_INTO_TRANSACTION = True
DELETE_EXISTING_TX_FOR_SAME_ORDERS = True  # DELETE ... WHERE order_id IN (...) AND data_source IN ('RLM','DTC24')
INSERT_BATCH_SIZE = 500

COMPANY_CODE = "ADAM_LIPPES"
# commercial_organisation : "US" si division_code == 10, sinon "OTHERS" (comme dans votre M)

# Jointure produits (optionnel) : barcodes -> product_id Shopify
# Si vide, aucune jointure ; sinon requête Supabase sur cette table (colonnes barcode, product_id).
PRODUCTS_BARCODE_TABLE = "products"  # ex: "products" ; mettre "" pour désactiver
PRODUCTS_BARCODE_COL = "barcode"
PRODUCTS_ID_COL = "product_id"

In [ ]:
import math
from datetime import date, datetime
from decimal import Decimal

import numpy as np
import pandas as pd

# Harmonisation des en-têtes (équivalent Power Query #"---- HARMONIZATION -----#")
_HARMO = {
    "Source.Name": "source_files",
    "Div #": "division_code",
    "Invoice #": "invoice_number",
    "Cust #": "customer_id",
    "Trans Type": "trans_type",
    "Div Name": "division_label",
    "Invoice Date": "date_origin",
    "Processed Date": "processed_date",
    "Sea": "season",
    "Customer Name": "customer_name",
    "Customer PO #": "customer_po",
    "Factor Code": "factor_code",
    "Duty % Factor": "duty_factor_rate",
    "Effective Prd/Year": "effective_prd_year",
    "Line #": "line",
    "STYLE": "style",
    "COLOR": "color",
    "COLOR Description": "color_description",
    "Units Sold": "quantity",
    "Merchandise Amount": "shop_amount",
    "Merch Disc %": "discount_rate",
    "Merchandise Amt. Net Of Merch. Disc.": "net_sales",
    "Freight Amount": "Shipping",
    "Tax Amount": "Taxes",
    "Gross Amount": "gross_amount",
    "Net After Terms Disc": "net_after_term_disc",
    "Gross Margin $": "gross_margin",
    "Gross Margin %": "gross_margin_rate",
    "MSRP": "msrp",
    "Unit Cost": "cogs_unit",
    "Extended Cost": "extended_cost",
    "Business Unit": "business_unit",
    "Business Manager": "business_manager",
    "Operations Manager": "operation_manager",
    "Operations Rep": "operation_rep",
    "Credit Memo Class": "credit_memo_class",
    "Credit Memo Sub Class": "credit_memo_subclass",
    "AR Collector": "ar_collector",
    "UPC": "barcode",
    "Date created": "created_at_timestamp",
    "Date modified": "updated_at_timestamp",
}


def _to_int(v, default=None):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return default
    if isinstance(v, (int, np.integer)):
        return int(v)
    s = str(v).strip().replace(",", "")
    if s == "" or s.lower() in ("nan", "none"):
        return default
    try:
        return int(float(s))
    except (TypeError, ValueError):
        return default


def _to_float(v, default=None):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return default
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v)
    s = str(v).strip().replace(",", "")
    if s == "" or s.lower() in ("nan", "none"):
        return default
    try:
        return float(s)
    except (TypeError, ValueError):
        return default


def _client_id_bigint(v) -> int:
    """Colonne client_id BIGINT : chiffres extraits, sinon déterministe sur le hash."""
    n = _to_int(v, None)
    if n is not None:
        return int(n) % (2**63 - 1) or 1
    s = str(v).strip() if v is not None else ""
    digits = "".join(ch for ch in s if ch.isdigit())
    if digits:
        return int(digits[:15]) % (2**63 - 1) or 1
    return (abs(hash(s)) % (2**63 - 1)) or 1


def _parse_date_parts(day, month, year):
    try:
        d, m, y = int(day), int(month), int(year)
        if y < 100:
            y += 2000
        return date(y, m, d)
    except Exception:
        return None


def harmonize_invoice_date(source_files: str, date_origin) -> date | None:
    """Reprend la logique #" Harmonization date (DD-MM ou MM-DD)" du M."""
    src = str(source_files or "").strip().lower()
    if pd.isna(date_origin) or date_origin is None or str(date_origin).strip() == "":
        return None
    if isinstance(date_origin, datetime):
        return date_origin.date()
    if isinstance(date_origin, date):
        return date_origin
    txt = str(date_origin).strip()
    for sep in ("/", ".", " "):
        txt = txt.replace(sep, "-")
    parts = [p for p in txt.split("-") if p != ""]
    if len(parts) < 2:
        return None
    farfetch_exact = src == "farfetch_invoices_2025.csv"
    if farfetch_exact:
        ypart = parts[2] if len(parts) > 2 else None
        return _parse_date_parts(parts[0], parts[1], ypart)
    ypart = parts[2] if len(parts) > 2 else None
    return _parse_date_parts(parts[1], parts[0], ypart)


def read_rlm_csv(path: Path) -> pd.DataFrame:
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            break
        except UnicodeDecodeError:
            df = None
    else:
        raise RuntimeError("Impossible de décoder le CSV (encodage).")
    df.columns = [str(c).strip() for c in df.columns]
    ren = {k: v for k, v in _HARMO.items() if k in df.columns}
    df = df.rename(columns=ren)
    required = {"source_files", "invoice_number", "trans_type", "shop_amount"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            "Colonnes attendues manquantes après harmonisation : "
            + ", ".join(sorted(missing))
            + ". Vérifiez les en-têtes du fichier (modèle Power Query)."
        )
    return df


def load_barcode_product_map(sb, table: str, bcol: str, pcol: str) -> dict[str, int]:
    if not table:
        return {}
    out = {}
    start = 0
    page = 1000
    while True:
        q = sb.table(table).select(f"{bcol},{pcol}").range(start, start + page - 1)
        res = q.execute()
        rows = res.data or []
        if not rows:
            break
        for r in rows:
            bc = r.get(bcol)
            pid = r.get(pcol)
            if bc is None or pid is None:
                continue
            key = str(bc).strip()
            if key:
                try:
                    out[key] = int(pid)
                except (TypeError, ValueError):
                    continue
        if len(rows) < page:
            break
        start += page
    return out


def dataframe_to_transaction_rows(
    df: pd.DataFrame,
    barcode_to_product: dict[str, int],
    company_code: str,
) -> list[dict]:
    rows_out: list[dict] = []
    for _, r in df.iterrows():
        src_file = str(r.get("source_files", "") or "")
        inv = _to_int(r.get("invoice_number"), None)
        if inv is None:
            continue
        order_id = inv
        trans = str(r.get("trans_type") or "").strip().upper()
        if trans == "INVOICE":
            acc_main = "Sales"
        elif trans == "CREDIT":
            acc_main = "Returns"
        elif trans == "VOID":
            acc_main = "Canceled"
        else:
            acc_main = "not_attributed"

        shop_amt = _to_float(r.get("shop_amount"), 0.0) or 0.0
        disc_rate = _to_float(r.get("discount_rate"), 0.0) or 0.0
        qty_base = _to_int(r.get("quantity"), 0) or 0
        data_source = "DTC24" if "DTC" in src_file.upper() else "RLM"
        if data_source in ("RLM", "DTC24"):
            qty_base = abs(int(qty_base))

        d_invoice = harmonize_invoice_date(src_file, r.get("date_origin"))
        if d_invoice is None:
            d_invoice = date.today()

        dt = datetime(d_invoice.year, d_invoice.month, d_invoice.day)
        division = _to_int(r.get("division_code"), None)
        commercial_org = "US" if division == 10 else "OTHERS"

        season = str(r.get("season") or "")
        cust_name = str(r.get("customer_name") or "")
        order_lbl = f"INV.{inv}".upper()
        tx_desc = f"{season}-{cust_name}-{order_lbl}"

        bc = str(r.get("barcode") or "").strip()
        product_id = barcode_to_product.get(bc)

        cogs_unit = _to_float(r.get("cogs_unit"), None)
        qty_for_cogs = qty_base

        def one_row(account_type: str, shop_amount: float, quantity: int):
            cogs_total = None
            if cogs_unit is not None and account_type in ("Sales", "Returns", "Canceled"):
                cogs_total = float(quantity) * float(cogs_unit)
                if account_type == "Returns" and cogs_total is not None:
                    cogs_total = -abs(cogs_total)
            return {
                "date": dt.isoformat(),
                "order_id": order_id,
                "client_id": _client_id_bigint(r.get("customer_id")),
                "account_type": account_type,
                "transaction_description": tx_desc,
                "shop_amount": float(shop_amount),
                "amount_currency": float(shop_amount),
                "transaction_currency": "USD",
                "location_id": None,
                "source_name": None,
                "status": "success",
                "product_id": product_id,
                "variant_id": product_id,
                "payment_method_name": "",
                "orders_details_id": None,
                "quantity": int(quantity),
                "exchange_rate": 0.0,
                "shop_currency": "USD",
                "cogs_unit": cogs_unit,
                "cogs_total": cogs_total,
                "data_source": data_source,
                "company_code": company_code,
                "commercial_organisation": commercial_org,
            }

        rows_out.append(one_row(acc_main, shop_amt, qty_base))

        disc_amt = shop_amt * (disc_rate / 100.0)
        if disc_amt != 0:
            rows_out.append(one_row("Discounts", -abs(disc_amt), qty_base))

    return rows_out


In [ ]:
import mimetypes

local = Path(LOCAL_FILE).expanduser().resolve()
assert local.is_file(), f"Fichier introuvable : {local}"
if UPLOAD_TO_STORAGE:
    assert BUCKET, "BUCKET ne peut pas être vide pour l'upload Storage"

dest = DEST_PATH or local.name
content_type = mimetypes.guess_type(str(local))[0] or "application/octet-stream"

print(f"Fichier  : {local}  ({local.stat().st_size / 1024:.1f} KB)")
print(f"Bucket   : {BUCKET!r} (upload={UPLOAD_TO_STORAGE})")
print(f"Dest     : {dest}")
print(f"Type     : {content_type}")


In [ ]:
# --- Storage (optionnel) ---
if UPLOAD_TO_STORAGE:
    existing = [b.name for b in supabase.storage.list_buckets()]
    if BUCKET not in existing:
        supabase.storage.create_bucket(BUCKET, options={"public": False})
        print(f"Bucket {BUCKET!r} créé")

    opts = {"content-type": content_type}
    if STORAGE_UPSERT:
        opts["upsert"] = "true"

    with open(local, "rb") as f:
        res = supabase.storage.from_(BUCKET).upload(path=dest, file=f, file_options=opts)
    print("Upload Storage terminé →", res)
else:
    print("Upload Storage ignoré (UPLOAD_TO_STORAGE=False)")

# --- Base : CSV → table `transaction` ---
if local.suffix.lower() == ".csv" and INSERT_INTO_TRANSACTION:
    df = read_rlm_csv(local)
    pmap = {}
    try:
        pmap = load_barcode_product_map(
            supabase,
            (PRODUCTS_BARCODE_TABLE or "").strip(),
            PRODUCTS_BARCODE_COL,
            PRODUCTS_ID_COL,
        )
    except Exception as e:
        print("Jointure produits ignorée (vérifiez le nom des colonnes / la table) :", e)
    if pmap:
        print(f"Jointure produits : {len(pmap)} codes-barres indexés")
    tx_rows = dataframe_to_transaction_rows(df, pmap, COMPANY_CODE)
    print(f"Lignes `transaction` à insérer : {len(tx_rows)}")

    order_ids = list({int(r["order_id"]) for r in tx_rows})
    if DELETE_EXISTING_TX_FOR_SAME_ORDERS and order_ids:
        for lo in range(0, len(order_ids), 200):
            chunk = order_ids[lo : lo + 200]
            supabase.table("transaction").delete().in_("order_id", chunk).in_(
                "data_source", ["RLM", "DTC24"]
            ).execute()
        print(f"Transactions RLM/DTC24 supprimées pour {len(order_ids)} order_id distincts")

    def _json_safe(obj):
        import math as _m

        if isinstance(obj, dict):
            return {k: _json_safe(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [_json_safe(v) for v in obj]
        if isinstance(obj, float) and (_m.isnan(obj) or _m.isinf(obj)):
            return None
        return obj

    bs = max(1, int(INSERT_BATCH_SIZE))
    for i in range(0, len(tx_rows), bs):
        batch = _json_safe(tx_rows[i : i + bs])
        supabase.table("transaction").insert(batch).execute()
        print(f"  Insert batch {i // bs + 1} ({len(batch)} lignes)")
    print("Import `transaction` terminé.")
elif local.suffix.lower() == ".csv" and not INSERT_INTO_TRANSACTION:
    print("CSV détecté mais INSERT_INTO_TRANSACTION=False — aucune écriture en base.")
else:
    print("Fichier non-CSV : aucune insertion `transaction`.")
